Note: For this final report, I'll be comparing baselines to roberta-large. I have left out the roberta-base models.

In [ ]:
# Setup and Imports
import os
import json
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from google.colab import drive

print(" Mounting Google Drive...")
drive.mount('/content/drive')

GDRIVE_PATH = '/content/drive/MyDrive/'
BASELINE_RESULTS_DIR = os.path.join(GDRIVE_PATH, 'Baseline_Test/')


ADVANCED_RESULTS_DIR = os.path.join(GDRIVE_PATH, 'LargeModels_Results/')
print(" Setup complete.")

In [ ]:
# Load results
def load_all_results(baseline_path, advanced_path):
    all_results = []
    classification_datasets = ["Essaysbig5", "GoEmotions"]


    if os.path.exists(baseline_path):
        for dataset_name in os.listdir(baseline_path):
            metric_file = os.path.join(baseline_path, dataset_name, 'metrics', 'baseline_metrics.json')
            if os.path.exists(metric_file):
                with open(metric_file, 'r') as f: data = json.load(f)
                best_model, score = max(data.items(), key=lambda item: item[1])
                model_label = f'Baseline ({best_model})'
                if dataset_name in classification_datasets:
                    all_results.append({'Dataset': dataset_name, 'Model Type': model_label, 'F1 Score': score, 'R2 Score': np.nan})
                else:
                    all_results.append({'Dataset': dataset_name, 'Model Type': model_label, 'F1 Score': np.nan, 'R2 Score': score})


    if os.path.exists(advanced_path):
        for dataset_name in os.listdir(advanced_path):
            metric_file = os.path.join(advanced_path, dataset_name, 'transformer_metrics.json')
            if os.path.exists(metric_file):
                with open(metric_file, 'r') as f: data = json.load(f)
                if 'eval_f1_weighted' in data:
                    all_results.append({'Dataset': dataset_name, 'Model Type': 'RoBERTa-large', 'F1 Score': data['eval_f1_weighted'], 'R2 Score': np.nan})
                elif 'eval_r2_score' in data:
                    all_results.append({'Dataset': dataset_name, 'Model Type': 'RoBERTa-large', 'R2 Score': data['eval_r2_score'], 'F1 Score': np.nan})

    return pd.DataFrame(all_results)

print(" Results loading function defined.")

In [ ]:
# Load, Display and Visualize
print("Loading and processing all saved metrics...")
final_comparison_df = load_all_results(BASELINE_RESULTS_DIR, ADVANCED_RESULTS_DIR)

if not final_comparison_df.empty:
    print("\n--- Final Performance Comparison ---")

    clf_df = final_comparison_df.dropna(subset=['F1 Score'])
    reg_df = final_comparison_df.dropna(subset=['R2 Score'])

    print("\nClassification Results (F1-Score):")
    print(clf_df.pivot(index='Dataset', columns='Model Type', values='F1 Score').round(4))

    print("\nRegression Results (R2-Score):")
    print(reg_df.pivot(index='Dataset', columns='Model Type', values='R2 Score').round(4))


    if not clf_df.empty:
        plt.figure(figsize=(12, 7))
        sns.barplot(data=clf_df, x='Dataset', y='F1 Score', hue='Model Type')
        plt.title('Classification Performance (F1 Score)', fontsize=16)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()


    if not reg_df.empty:
        plt.figure(figsize=(12, 7))
        sns.barplot(data=reg_df, x='Dataset', y='R2 Score', hue='Model Type')
        plt.title('Regression Performance (R2 Score)', fontsize=16)
        plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.tight_layout()
        plt.show()
else:
    print("\nNo results found to display.")
